In [1]:
# ==================== INSTALL REQUIRED PACKAGES ====================
!pip install -q biopython plotly pandas fair-esm torch torchvision torchaudio

import torch
import esm
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries installed successfully!")
print(f"GPU Available: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 6.1 MB/s eta 0:00:00
✅ All libraries installed successfully!
GPU Available: True


In [3]:
# ====================== HEAVY AI PROTEIN FOLDING ANALYZER ======================

print("🧠 Heavy AI Protein Folding Analyzer (ESM-2 Powered)")
print("="*70)

# Input Sequence
sequence = input("Paste your protein sequence here: ").strip().upper()

# Clean sequence
valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
seq = "".join([aa for aa in sequence if aa in valid_aa])

if len(seq) < 20:
    print("❌ Error: Sequence must be at least 20 residues long.")
else:
    if len(seq) > 1022:
        print("⚠️ Sequence too long. Truncating to 1022 residues (ESM-2 limit).")
        seq = seq[:1022]

    print(f"✅ Analyzing sequence of length: {len(seq)}")

    # Basic Analysis
    analysis = ProteinAnalysis(seq)

    # ====================== LOAD ESM-2 MODEL ======================
    print("\n🔄 Loading ESM-2 model (650M)... This may take a while on first run.")
    model_name = "esm2_t33_650M_UR50D"
    model, alphabet = esm.pretrained.load_model_and_alphabet(model_name)
    model.eval()

    if torch.cuda.is_available():
        model = model.cuda()
        device = "cuda"
        print("🚀 Using GPU")
    else:
        device = "cpu"
        print("⚡ Using CPU (slower)")

    # ====================== COMPUTE ESM-2 EMBEDDINGS ======================
    print("🧬 Computing ESM-2 embeddings and contact map...")
    batch_converter = alphabet.get_batch_converter()
    data = [("protein", seq)]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)

    with torch.no_grad():
        batch_tokens = batch_tokens.to(device)
        results = model(batch_tokens, repr_layers=[33], return_contacts=True)

    token_representations = results["representations"][33][0]
    mean_embedding = token_representations.mean(dim=0).cpu().numpy()
    embedding_norm = torch.norm(token_representations.mean(dim=0)).item()

    contacts = results["contacts"][0].cpu().numpy()

    # ====================== HELPER FUNCTIONS ======================
    def count_fraction(seq_str, residues):
        return round(sum(seq_str.count(aa) for aa in residues) / len(seq_str) * 100, 2)

    # ====================== 20 HEAVY AI METRICS ======================
    metrics = {
        "Sequence Length": len(seq),
        "ESM-2 Embedding Norm": round(embedding_norm, 4),
        "Hydrophobic Fraction (%)": count_fraction(seq, 'VILFMWA'),
        "Charged Fraction (%)": count_fraction(seq, 'KRDEH'),
        "Polar Fraction (%)": count_fraction(seq, 'STNQYC'),
        "Disorder-Promoting (%)": count_fraction(seq, 'PGSTQEDKR'),
        "GRAVY Score": round(analysis.gravy(), 3),
        "Instability Index": round(analysis.instability_index(), 2),
        "Isoelectric Point (pI)": round(analysis.isoelectric_point(), 2),
        "Aliphatic Index": round((seq.count('A') + 2.9*seq.count('V') + 3.9*(seq.count('I')+seq.count('L'))) / len(seq) * 100, 2),
        "Helix Propensity (%)": round(analysis.secondary_structure_fraction()[0]*100, 2),
        "Cysteine Count": seq.count('C'),
        "Proline Count": seq.count('P'),
        "Glycine Count": seq.count('G'),
        "Net Charge at pH 7.0": round(analysis.charge_at_pH(7.0), 2),
        "Aromaticity": round(analysis.aromaticity(), 4),
        "Contact Density (ESM-2)": round(contacts.mean(), 4),
        "Low Complexity Score": round((seq.count('P') + seq.count('G') + seq.count('S')*2) / len(seq), 3),
        "AI Folding Readiness Score": round(embedding_norm * (1 - analysis.gravy()/10), 3)
    }

    # Display Metrics
    df = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])
    print("\n" + "="*60)
    print("🔥 20 ADVANCED AI / DEEP LEARNING METRICS")
    print("="*60)
    display(df)

    # ====================== VISUALIZATIONS ======================
    print("\n📊 Generating Visualizations...")

    # 1. Contact Map
    fig1 = px.imshow(contacts, title="ESM-2 Predicted Contact Map",
                     color_continuous_scale='viridis', aspect='auto')
    fig1.show()

    # 2. Hydrophobicity Profile
    hydro = analysis.protein_scale("KyteDoolittle", 9)
    fig2 = px.line(y=hydro, title="Hydrophobicity Profile (Key for Folding)",
                   labels={"index": "Position", "value": "Score"})
    fig2.show()

    # 3. Amino Acid Composition
    aa_count = analysis.count_amino_acids()
    aa_df = pd.DataFrame.from_dict(aa_count, orient='index', columns=['Count'])
    aa_df['Percentage'] = (aa_df['Count'] / len(seq) * 100).round(2)
    aa_df = aa_df.sort_values('Percentage', ascending=False)

    fig3 = px.bar(aa_df, x=aa_df.index, y='Percentage', title="Amino Acid Composition (%)")
    fig3.show()

    # Save results
    df.to_csv(f"ai_protein_metrics_{len(seq)}aa.csv", index=False)
    print(f"\n✅ Analysis completed! Results saved as 'ai_protein_metrics_{len(seq)}aa.csv'")

🧠 Heavy AI Protein Folding Analyzer (ESM-2 Powered)
Paste your protein sequence here: MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESHAG
✅ Analyzing sequence of length: 85

🔄 Loading ESM-2 model (650M)... This may take a while on first run.


OutOfMemoryError: CUDA out of memory. Tried to allocate 26.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 19.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.31 GiB is allocated by PyTorch, and 106.81 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [4]:
# ==================== INSTALLATION ====================
!pip install -q biopython plotly pandas fair-esm

import torch
import esm
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import plotly.express as px
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries installed!")
print(f"GPU Available: {torch.cuda.is_available()}")

✅ Libraries installed!
GPU Available: True


In [5]:
# ====================== MEMORY-EFFICIENT HEAVY AI ANALYZER ======================

print("🧠 Heavy AI Protein Folding Analyzer (Optimized)")
print("="*75)

# ====================== INPUT ======================
sequence = input("Paste your protein sequence here: ").strip().upper()

valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
seq = "".join(aa for aa in sequence if aa in valid_aa)

if len(seq) < 20:
    print("❌ Sequence too short!")
else:
    if len(seq) > 800:
        seq = seq[:800]
        print(f"⚠️ Sequence truncated to 800 residues for memory safety.")

    print(f"✅ Analyzing sequence: {len(seq)} residues")

    analysis = ProteinAnalysis(seq)

    # ====================== CHOOSE SMALLER MODEL TO AVOID OOM ======================
    print("\n🔄 Loading optimized ESM-2 model...")

    # Using smaller but still powerful model to prevent CUDA OOM
    model_name = "esm2_t12_35M_UR50D"   # Much lighter than 650M

    model, alphabet = esm.pretrained.load_model_and_alphabet(model_name)
    model.eval()

    # Memory optimization
    if torch.cuda.is_available():
        try:
            model = model.cuda()
            device = "cuda"
            print("🚀 Using GPU (Optimized)")
        except:
            model = model.cpu()
            device = "cpu"
            print("⚠️ GPU ran out of memory. Falling back to CPU.")
    else:
        device = "cpu"
        print("⚡ Running on CPU")

    # ====================== COMPUTE ESM-2 FEATURES ======================
    print("🧬 Computing embeddings and contact map...")

    batch_converter = alphabet.get_batch_converter()
    data = [("protein", seq)]
    _, _, batch_tokens = batch_converter(data)

    with torch.no_grad():
        batch_tokens = batch_tokens.to(device)
        results = model(batch_tokens, repr_layers=[12], return_contacts=True)

        token_representations = results["representations"][12][0]
        mean_embedding = token_representations.mean(dim=0).cpu().numpy()
        embedding_norm = float(torch.norm(token_representations.mean(dim=0)).item())

        contacts = results["contacts"][0].cpu().numpy()

    # ====================== 20 AI METRICS ======================
    def count_fraction(s, residues):
        return round(sum(s.count(aa) for aa in residues) / len(s) * 100, 2)

    metrics = {
        "Sequence Length": len(seq),
        "ESM Model Used": model_name,
        "ESM Embedding Norm": round(embedding_norm, 4),
        "Hydrophobic Fraction (%)": count_fraction(seq, 'VILFMWA'),
        "Charged Fraction (%)": count_fraction(seq, 'KRDEH'),
        "Polar Fraction (%)": count_fraction(seq, 'STNQYC'),
        "Disorder-Promoting (%)": count_fraction(seq, 'PGSTQEDKR'),
        "GRAVY Score": round(analysis.gravy(), 3),
        "Instability Index": round(analysis.instability_index(), 2),
        "Isoelectric Point (pI)": round(analysis.isoelectric_point(), 2),
        "Aliphatic Index": round((seq.count('A') + 2.9*seq.count('V') + 3.9*(seq.count('I')+seq.count('L'))) / len(seq) * 100, 2),
        "Helix Propensity (%)": round(analysis.secondary_structure_fraction()[0]*100, 2),
        "Cysteine Count": seq.count('C'),
        "Proline Count": seq.count('P'),
        "Glycine Count": seq.count('G'),
        "Net Charge at pH 7": round(analysis.charge_at_pH(7.0), 2),
        "Contact Density (ESM)": round(contacts.mean(), 4),
        "AI Folding Readiness": round(embedding_norm * (1 - analysis.gravy()/10), 3)
    }

    df = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])
    display(df)

    # ====================== VISUALIZATIONS ======================
    print("\n📊 Generating Plots...")

    fig1 = px.imshow(contacts, title="ESM-2 Contact Map", color_continuous_scale='viridis')
    fig1.show()

    hydro = analysis.protein_scale("KyteDoolittle", 9)
    fig2 = px.line(y=hydro, title="Hydrophobicity Profile")
    fig2.show()

    # Amino Acid Composition
    aa_count = analysis.count_amino_acids()
    aa_df = pd.DataFrame.from_dict(aa_count, orient='index', columns=['Count'])
    aa_df['Percentage'] = (aa_df['Count'] / len(seq) * 100).round(2)
    aa_df = aa_df.sort_values('Percentage', ascending=False)
    fig3 = px.bar(aa_df, x=aa_df.index, y='Percentage', title="Amino Acid Composition (%)")
    fig3.show()

    # Save
    df.to_csv(f"ai_folding_metrics_{len(seq)}aa.csv", index=False)
    print(f"\n✅ Analysis Completed & Saved!")

🧠 Heavy AI Protein Folding Analyzer (Optimized)
Paste your protein sequence here: MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESHAG
✅ Analyzing sequence: 85 residues

🔄 Loading optimized ESM-2 model...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D-contact-regression.pt
🚀 Using GPU (Optimized)
🧬 Computing embeddings and contact map...


,Metric,Value
0,Sequence Length,85
1,ESM Model Used,esm2_t12_35M_UR50D
2,ESM Embedding Norm,6.414
3,Hydrophobic Fraction (%),43.53
4,Charged Fraction (%),29.41
5,Polar Fraction (%),22.35
6,Disorder-Promoting (%),44.71
7,GRAVY Score,-0.039
8,Instability Index,33.24
9,Isoelectric Point (pI),6.4



📊 Generating Plots...


TypeError: string indices must be integers, not 'str'

In [6]:
!pip install -q biopython plotly pandas fair-esm

import torch
import esm
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils import ProtParamData   # ← Important fix
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries installed successfully!")

✅ Libraries installed successfully!


In [7]:
# ====================== FIXED HEAVY AI PROTEIN ANALYZER ======================

print("🧠 Heavy AI Protein Folding Analyzer (ESM-2 + Fixed)")
print("="*80)

# ====================== INPUT ======================
sequence = input("Paste your protein sequence here: ").strip().upper()

valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
seq = "".join(aa for aa in sequence if aa in valid_aa)

if len(seq) < 20:
    print("❌ Sequence too short!")
else:
    if len(seq) > 800:
        seq = seq[:800]
        print(f"⚠️ Sequence truncated to 800 residues for stability.")

    print(f"✅ Analyzing {len(seq)} residue protein")

    analysis = ProteinAnalysis(seq)

    # ====================== LOAD LIGHT ESM-2 MODEL ======================
    print("\n🔄 Loading ESM-2 model...")
    model_name = "esm2_t12_35M_UR50D"
    model, alphabet = esm.pretrained.load_model_and_alphabet(model_name)
    model.eval()

    if torch.cuda.is_available():
        try:
            model = model.cuda()
            device = "cuda"
            print("🚀 GPU Activated")
        except:
            device = "cpu"
            print("⚠️ Falling back to CPU")
    else:
        device = "cpu"

    # ====================== ESM-2 COMPUTATION ======================
    print("🧬 Computing ESM-2 embeddings...")
    batch_converter = alphabet.get_batch_converter()
    data = [("protein", seq)]
    _, _, batch_tokens = batch_converter(data)

    with torch.no_grad():
        batch_tokens = batch_tokens.to(device)
        results = model(batch_tokens, repr_layers=[12], return_contacts=True)

        token_reps = results["representations"][12][0]
        embedding_norm = float(torch.norm(token_reps.mean(dim=0)).item())
        contacts = results["contacts"][0].cpu().numpy()

    # ====================== HELPER FUNCTIONS ======================
    def count_fraction(s, residues):
        return round(sum(s.count(aa) for aa in residues) / len(s) * 100, 2)

    # ====================== 20+ AI METRICS ======================
    metrics = {
        "Sequence Length": len(seq),
        "ESM Model": model_name,
        "ESM Embedding Norm": round(embedding_norm, 4),
        "Hydrophobic Fraction (%)": count_fraction(seq, 'VILFMWA'),
        "Charged Fraction (%)": count_fraction(seq, 'KRDEH'),
        "Polar Fraction (%)": count_fraction(seq, 'STNQYC'),
        "Disorder-Promoting (%)": count_fraction(seq, 'PGSTQEDKR'),
        "GRAVY Score": round(analysis.gravy(), 3),
        "Instability Index": round(analysis.instability_index(), 2),
        "Isoelectric Point (pI)": round(analysis.isoelectric_point(), 2),
        "Aliphatic Index": round((seq.count('A') + 2.9*seq.count('V') + 3.9*(seq.count('I')+seq.count('L'))) / len(seq) * 100, 2),
        "Helix Propensity (%)": round(analysis.secondary_structure_fraction()[0]*100, 2),
        "Cysteine Count": seq.count('C'),
        "Proline Count": seq.count('P'),
        "Glycine Count": seq.count('G'),
        "Net Charge pH 7.0": round(analysis.charge_at_pH(7.0), 2),
        "Contact Density (ESM)": round(contacts.mean(), 4),
        "AI Folding Readiness": round(embedding_norm * (1 - analysis.gravy()/10), 3)
    }

    df = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])
    display(df)

    # ====================== VISUALIZATIONS (6 GRAPHS TOTAL) ======================
    print("\n📊 Generating 6 Professional Graphs...")

    # Graph 1: ESM-2 Contact Map
    fig1 = px.imshow(contacts, title="ESM-2 Predicted Contact Map",
                     color_continuous_scale='viridis')
    fig1.show()

    # Graph 2: Hydrophobicity Profile (FIXED)
    hydro = analysis.protein_scale(ProtParamData.kd, 9)   # ← Fixed Here
    fig2 = px.line(y=hydro, title="Hydrophobicity Profile (Kyte-Doolittle)",
                   labels={"index": "Residue Position", "value": "Score"})
    fig2.show()

    # Graph 3: Net Charge vs pH
    phs = list(range(0, 15))
    charges = [analysis.charge_at_pH(p) for p in phs]
    fig3 = px.line(x=phs, y=charges, title="Net Charge vs pH", markers=True)
    fig3.show()

    # Graph 4: Amino Acid Composition
    aa_count = analysis.count_amino_acids()
    aa_df = pd.DataFrame.from_dict(aa_count, orient='index', columns=['Count'])
    aa_df['Percentage'] = (aa_df['Count'] / len(seq) * 100).round(2)
    aa_df = aa_df.sort_values('Percentage', ascending=False)
    fig4 = px.bar(aa_df, x=aa_df.index, y='Percentage', title="Amino Acid Composition (%)")
    fig4.show()

    # ==================== 4 NEW GRAPHS ====================

    # Graph 5: Charge Distribution (New)
    fig5 = go.Figure()
    fig5.add_trace(go.Bar(x=list(aa_count.keys()), y=[analysis.charge_at_pH(7) * (aa_count[aa]/len(seq)) for aa in aa_count.keys()],
                         name="Charge Contribution"))
    fig5.update_layout(title="Amino Acid Charge Contribution at pH 7")
    fig5.show()

    # Graph 6: Cumulative Hydrophobicity (New)
    cum_hydro = pd.Series(hydro).cumsum()
    fig6 = px.line(y=cum_hydro, title="Cumulative Hydrophobicity Along Sequence")
    fig6.show()

    # Graph 7: Secondary Structure Propensity (New)
    helix, turn, sheet = analysis.secondary_structure_fraction()
    fig7 = px.pie(values=[helix, turn, sheet],
                  names=['Helix', 'Turn', 'Sheet'],
                  title="Predicted Secondary Structure Composition")
    fig7.show()

    # Save Results
    df.to_csv(f"heavy_ai_protein_analysis_{len(seq)}aa.csv", index=False)
    print(f"\n✅ Analysis Complete! File saved: heavy_ai_protein_analysis_{len(seq)}aa.csv")

🧠 Heavy AI Protein Folding Analyzer (ESM-2 + Fixed)
Paste your protein sequence here: MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESHAG
✅ Analyzing 85 residue protein

🔄 Loading ESM-2 model...
🚀 GPU Activated
🧬 Computing ESM-2 embeddings...


,Metric,Value
0,Sequence Length,85
1,ESM Model,esm2_t12_35M_UR50D
2,ESM Embedding Norm,6.414
3,Hydrophobic Fraction (%),43.53
4,Charged Fraction (%),29.41
5,Polar Fraction (%),22.35
6,Disorder-Promoting (%),44.71
7,GRAVY Score,-0.039
8,Instability Index,33.24
9,Isoelectric Point (pI),6.4



📊 Generating 6 Professional Graphs...



✅ Analysis Complete! File saved: heavy_ai_protein_analysis_85aa.csv
